# Runge–Kutta Methods and `rk_functions.py`

## Purpose and scope

The companion module [`rk_functions.py`](rk_functions.py) contains routines for advancing a two-dimensional gravitational orbit with a fourth-order Runge–Kutta method and an attempted adaptive-step wrapper.

This notebook explains the mathematics and maps it onto the organization of those routines. It intentionally contains no executable Python code: the goal is to understand the algorithm before using or modifying its implementation.

The central questions are:

- How is a higher-order differential equation rewritten as a first-order system?
- How does RK4 combine several derivative estimates within one timestep?
- Why can two half-steps be compared with one full step to estimate error?
- What does an adaptive integrator need to communicate to its caller?
- Which parts of the current module are general, and which are specific to gravity?

## Ordinary differential equations as state evolution

A Runge–Kutta method solves an initial-value problem written in first-order form,

$$\frac{d\mathbf{x}}{dt}=\mathbf{f}(t,\mathbf{x}),
\qquad \mathbf{x}(t_0)=\mathbf{x}_0.$$

Here $\mathbf{x}$ is the **state vector** and $\mathbf{f}$ returns its time derivative. The numerical task is to approximate the state at a sequence of later times.

For two-dimensional motion, position and velocity can be combined into

$$\mathbf{x}=(r_x,r_y,v_x,v_y).$$

The derivative of this state is

$$\frac{d\mathbf{x}}{dt}
=\left(v_x,v_y,a_x,a_y\right).$$

This conversion is fundamental: Runge–Kutta methods do not treat position and velocity as separate kinds of variables. They advance every component of one first-order state using the same stage structure.

## The gravitational derivative function

The function `gravrk` supplies $\mathbf{f}(t,\mathbf{x})$ for a particle moving in a central inverse-square gravitational field. The parameter array contains $GM$, the product of the gravitational constant and central mass.

With

$$r=\sqrt{r_x^2+r_y^2},$$

the acceleration is

$$a_x=-\frac{GM}{r^3}r_x,
\qquad
a_y=-\frac{GM}{r^3}r_y.$$

Therefore `gravrk` places the following four values in its derivative array:

$$\mathbf{f}(t,\mathbf{x})=
\left(v_x,v_y,-\frac{GM}{r^3}r_x,-\frac{GM}{r^3}r_y\right).$$

Although time is accepted as an argument, this particular force law has no explicit time dependence. The state still changes with time, so the acceleration changes as the orbiting object moves.

The equations are singular at $r=0$. The present derivative routine does not detect or regularize a collision with the central body.

## Why Euler's method is not enough

Forward Euler uses one derivative evaluation:

$$\mathbf{x}_{n+1}=\mathbf{x}_n+h\mathbf{f}(t_n,\mathbf{x}_n).$$

It follows the tangent at the beginning of the step and has global error $O(h)$. In an orbit, the direction and magnitude of acceleration vary continuously. A single beginning-of-step slope can therefore accumulate substantial phase and energy errors.

Runge–Kutta methods sample the derivative at several carefully chosen trial states inside the timestep. These trial states are not additional physical measurements; they are numerical probes used to estimate how the slope changes between $t_n$ and $t_n+h$.

## Classical fourth-order Runge–Kutta method

For a step of length $h$, classical RK4 constructs four stage derivatives:

$$\mathbf{k}_1=\mathbf{f}(t_n,\mathbf{x}_n),$$

$$\mathbf{k}_2=\mathbf{f}\left(t_n+\frac{h}{2},
\mathbf{x}_n+\frac{h}{2}\mathbf{k}_1\right),$$

$$\mathbf{k}_3=\mathbf{f}\left(t_n+\frac{h}{2},
\mathbf{x}_n+\frac{h}{2}\mathbf{k}_2\right),$$

$$\mathbf{k}_4=\mathbf{f}\left(t_n+h,
\mathbf{x}_n+h\mathbf{k}_3\right).$$

The new state is the weighted average

$$\mathbf{x}_{n+1}=\mathbf{x}_n+
\frac{h}{6}\left(\mathbf{k}_1+2\mathbf{k}_2+2\mathbf{k}_3+\mathbf{k}_4\right).$$

The weights give the two midpoint slopes twice the influence of the endpoint slopes. The result has local truncation error $O(h^5)$ and accumulated global error $O(h^4)$ for a sufficiently smooth problem.

## How `rk4` represents the four stages

The arrays named `F1`, `F2`, `F3`, and `F4` correspond to $\mathbf{k}_1$ through $\mathbf{k}_4$. The temporary array `xtemp` holds each trial state.

| Mathematical stage | Time used | Trial state used | Module array |
|---|---:|---|---|
| $\mathbf{k}_1$ | $t_n$ | $\mathbf{x}_n$ | `F1` |
| $\mathbf{k}_2$ | $t_n+h/2$ | $\mathbf{x}_n+(h/2)\mathbf{k}_1$ | `F2` |
| $\mathbf{k}_3$ | $t_n+h/2$ | $\mathbf{x}_n+(h/2)\mathbf{k}_2$ | `F3` |
| $\mathbf{k}_4$ | $t_n+h$ | $\mathbf{x}_n+h\mathbf{k}_3$ | `F4` |

After forming the weighted average, `rk4` overwrites the supplied state array `x` with the new state. This is an **in-place update**: the previous state is no longer present in that array unless the caller saved a copy.

The argument `nX` specifies the number of state components. In the gravitational application it should be 4 and must agree with the lengths of the supplied state and derivative arrays.

## Fixed step size: accuracy and cost

One RK4 step requires four derivative evaluations. For a fixed interval, halving $h$ therefore roughly doubles the number of steps and derivative evaluations. In the asymptotic regime, however, the global error should decrease by approximately

$$2^4=16.$$

A timestep study can test that expectation:

1. solve the same problem with $h$, $h/2$, and $h/4$;
2. compare a meaningful observable, such as final position or orbital period; and
3. verify that successive differences decrease consistently with fourth-order convergence.

RK4 is accurate and broadly useful, but it is not a symplectic integrator. Over very long orbital integrations, small energy and phase errors can accumulate systematically. Method choice should depend on the observable and duration of interest, not only on formal order.

## Adaptive timesteps by step doubling

The function `rka` attempts to choose a suitable timestep automatically. For a proposed step $h$, it computes the endpoint in two ways:

- `xBig`: one RK4 step of length $h$;
- `xSmall`: two successive RK4 steps of length $h/2$.

Both calculations begin from the same state. Because RK4 has local error proportional to $h^5$, the two-half-step result is normally more accurate. If a single-step local error is approximately $Ch^5$, then the combined error from two half-steps scales as

$$2C\left(\frac{h}{2}\right)^5=\frac{Ch^5}{16}.$$

The difference between the two results is therefore related to the leading local error. In a standard step-doubling derivation, the more accurate result's error is estimated from this difference with an additional factor of $1/15$.

The current routine uses the raw difference rather than dividing by 15. This is conservative as an acceptance test, but its `err` parameter should not be interpreted as a precisely calibrated local-error tolerance.

## Scaling a multicomponent error

Position and velocity components have different units and magnitudes, so an absolute difference cannot be compared directly across the state vector. The routine forms, for each component,

$$R_i=\frac{|x_{i,\mathrm{small}}-x_{i,\mathrm{big}}|}
{\texttt{err}\left(|x_{i,\mathrm{small}}|+|x_{i,\mathrm{big}}|\right)/2+\epsilon},$$

and chooses

$$R=\max_i R_i.$$

The step is accepted when $R<1$. Using the maximum means that every state component must satisfy the requested relative scale.

The small constant $\epsilon$ prevents division by zero, but it is not a physically meaningful absolute tolerance. Near a zero crossing, a robust adaptive solver normally combines separate relative and absolute tolerances,

$$\text{scale}_i=\text{atol}_i+	ext{rtol}_i|x_i|.$$

That distinction matters for orbital components that naturally pass through zero.

## Choosing the next timestep

If the normalized error ratio is $R$, fifth-order local-error scaling suggests

$$h_{\mathrm{new}}=S\,h_{\mathrm{old}}R^{-1/5},$$

where $S<1$ is a safety factor. The routine uses $S=0.9$ and exponent $-0.20=-1/5$.

The intended behavior is:

- if $R>1$, reject the trial and retry with a smaller timestep;
- if $R<1$, accept the more accurate two-half-step result; and
- limit how dramatically the timestep may shrink or grow in one attempt.

Step-size limits are important because a noisy error estimate can otherwise request an extreme change. They should be proportional to the old timestep, for example

$$0.2h_{\mathrm{old}}\leq h_{\mathrm{new}}
\leq5h_{\mathrm{old}}.$$

## Important limitations of the current adaptive routine

The present `rka` function demonstrates the idea of step doubling, but it should not yet be treated as a complete adaptive integrator.

1. **The timestep clamps are incorrect.** The current expressions compare with `tau_old/safe2` and `safe2/tau_old`. The second has inverse-time dimensions, and the resulting assignments do not implement the intended range $0.2h_{\mathrm{old}}$ to $5h_{\mathrm{old}}$.

2. **Updated time and timestep are not returned.** Python numbers are immutable, so changing local variables `t` and `tau` inside `rka` does not update the caller's values. The state may advance by a different accepted step without the caller knowing the accepted endpoint time or suggested next step.

3. **The derivative argument is not used.** Both `rk4` and `rka` accept an argument named `deriv`, but `rk4` calls `gravrk` directly. The implementation is therefore hard-wired to this gravitational model rather than being a general ODE solver.

4. **Failure is only printed.** After 100 unsuccessful attempts, the routine prints a message but does not raise an exception or return a status that the caller must handle.

5. **No Richardson correction is applied.** The accepted two-half-step state is more accurate than the full-step state, but the available difference could also be used to improve the estimate further.

These are software-interface and implementation issues, not failures of the Runge–Kutta mathematics itself.

## What a complete adaptive interface should provide

A usable adaptive stepper should communicate at least three results:

$$\left(\mathbf{x}_{\mathrm{accepted}},
h_{\mathrm{accepted}},h_{\mathrm{next}}\right).$$

The caller can then advance time consistently,

$$t\leftarrow t+h_{\mathrm{accepted}},$$

and use $h_{\mathrm{next}}$ for the next proposal. It should also report failure explicitly and accept a derivative function as an argument so the numerical algorithm is independent of the physical model.

This separation of responsibilities is valuable:

- the derivative function defines the model;
- the RK stage routine defines one numerical step;
- the adaptive controller accepts or rejects a proposed step; and
- the outer integration loop manages time, output, stopping events, and diagnostics.

## Summary

- A higher-order mechanical problem becomes a first-order system by including both position and velocity in the state.
- Classical RK4 samples four slopes and combines them to achieve fourth-order global accuracy.
- `gravrk` defines the gravitational model, while `rk4` advances its four-component state in place.
- Step doubling compares one full RK4 step with two half-steps and provides an estimate of local error.
- Adaptive control requires meaningful error scaling, accept/reject logic, bounded timestep changes, and a clear return interface.
- The fixed-step `rk4` routine represents the standard RK4 algorithm, but the current `rka` routine needs corrections before it can safely manage adaptive time evolution.

The broader modeling lesson is that a correct mathematical formula is only one part of a trustworthy computational method. State mutation, units, error definitions, return values, failure behavior, and convergence tests are equally important.